# 03. 공유 메모리: 에이전트들이 결과를 이어받는 방법

01과 02에서 에이전트 프롬프트 앞에 `Context from previous work:`가 계속 붙어 있었습니다. 이 노트북에서는 그 내용이
**어디에 저장되고, 어떻게 골라져서, 누가 읽을 수 있는지** 직접 열어 봅니다.

**이런 분께 맞습니다**

- "에이전트가 지난 작업을 기억한다"는 게 정확히 무슨 뜻인지 알고 싶은 분
- 대화형으로 쓰는 Claude Code 같은 도구와 syncmill이 기억을 나누게 하고 싶은 분
- 메모리에 무엇이 남는지(그리고 무엇을 넣으면 안 되는지) 알아 두고 싶은 분

**이 노트북에서 해 보는 것**

| 장 | 하는 일 | 끝나면 알게 되는 것 |
|---|---|---|
| 0 | 공유 메모리가 필요한 이유 | 무엇을 해 주고, 무엇은 해 주지 않는지 |
| 1~2 | 두 번 실행하고 메모리 파일 열어 보기 | 기록이 저장되는 위치와 형식, 태그의 뜻 |
| 3 | 다음 작업 프롬프트에 붙는 모습 | 관련 기록을 고르는 방식과 한계 |
| 4 | 전략의 중간 산출물 | 종류 태그와 단계별로 읽는 기록 |
| 5 | `status`, 메모리가 고장 났을 때 | 상태 확인과 "있으면 좋은 기능"이라는 의미 |
| 6 | `serve`로 다른 도구에서 읽기 | 읽기 전용 MCP 게이트웨이 사용법 |
| 7 | 대화형 세션과 협업, 실무 주의점 | 같은 저장소를 가리키게 하는 방법, 민감 정보 |

**비용과 안전**: 01과 같은 샌드박스에서 가짜 에이전트로 실행합니다. 실제 AI 호출은 없습니다. 소요 시간은 약 10분입니다.

## 0. 왜 공유 메모리가 필요한가요?

AI 코딩 에이전트는 기본적으로 **세션이 끝나면 잊습니다.** 또 Codex가 알아낸 것을 Kimi Code는 모릅니다. 그래서 사람이
같은 배경 설명을 매번 다시 적게 됩니다.

syncmill의 공유 메모리는 이 틈을 메웁니다.

- 실행이 **성공하면** 그 결과(요청, 에이전트, 답변, 바뀐 파일 목록)를 기록합니다.
- 다음 작업을 시작할 때 **요청과 관련 있는 기록을 찾아** 프롬프트 앞에 붙입니다.
- 에이전트끼리는 직접 대화하지 않습니다. 메모리를 읽고 쓰는 것은 **syncmill(감독자)만** 합니다.

반대로, 공유 메모리가 **하지 않는 일**도 알아 두세요.

- **코드를 합치지 않습니다.** 기록되는 것은 에이전트의 답변(설명)이지 코드 자체가 아닙니다. 코드는 패치와 `apply`로만
  움직입니다.
- **빠짐없이 전달된다는 보장이 없습니다.** 관련도 검색으로 상위 몇 개만, 정해진 분량 안에서 붙습니다.
- **반드시 있어야 하는 기능이 아닙니다.** 메모리가 고장 나도 작업은 계속됩니다(5장).

## 1. 준비: 두 번 실행해서 기록 만들기

샌드박스를 만들고, 서로 다른 에이전트로 두 번 실행합니다. 첫 번째는 설치 방법, 두 번째는 사용 예시를 쓰는 작업입니다.

In [1]:
from _sandbox import Sandbox, syncmill_executable

box = Sandbox.create()
box.init_project()

box.set_scenario({"codex": {"files": {"INSTALL.md": "pip install textstats\n"},
                            "message": "설치 방법: pip install textstats 명령으로 설치합니다."}})
first = box.sm("run", "README에 설치 방법을 추가해 줘", "--agent", "codex", show=False)

box.set_scenario({"kimi-code": {"files": {"USAGE.md": "count_words('a b')\n"},
                                "message": "사용 예시: count_words('a b')는 2를 돌려줍니다."}})
second = box.sm("run", "README에 사용 예시를 추가해 줘", "--agent", "kimi-code", show=False)
box.set_scenario(None)

print("실행 1:", box.run_id(first))
print("실행 2:", box.run_id(second))

실행 1: ba9ec6c8-ec46-4838-8805-f67f66a53ce6
실행 2: 909beb42-ef4c-46c7-b540-e123f65a6b74


## 2. 기록은 어디에, 어떤 모양으로 저장되나요?

기본 메모리 백엔드(memtomem)는 기록을 **사람이 읽을 수 있는 Markdown 파일**로 남기고, 검색용 색인을 SQLite DB에 둡니다.

- `init`한 실제 프로젝트에서는 저장소 안 `.syncmill/memory/`에 저장됩니다.
- 이 샌드박스에서는 `state/memory/files/`에 저장되도록 경로를 고정해 두었습니다.

In [2]:
memory_files = box.state / "memory" / "files"
for path in sorted(memory_files.glob("*.md")):
    print(path.name)

2026-10-14.agent-runtime.codex.md
2026-10-14.agent-runtime.kimi-code.md
2026-10-14.shared.md


**출력 읽는 법** — 파일 이름은 `{날짜}.{namespace}.md`입니다(namespace의 `:`는 `.`로 바뀜).
(같은 폴더에 `.`으로 시작하는 `*.lock` 파일도 있는데, 여러 프로세스가 동시에 기록할 때 서로 덮어쓰지 않게 하는 잠금 파일입니다.)

| 파일 | namespace | 뜻 |
|---|---|---|
| `….agent-runtime.codex.md` | `agent-runtime:codex` | codex의 **개인 기록**. codex가 만든 결과의 원본 |
| `….agent-runtime.kimi-code.md` | `agent-runtime:kimi-code` | kimi-code의 개인 기록 |
| `….shared.md` | `shared` | **모든 에이전트가 함께 보는 공동 기록**. 개인 기록의 복사본 |

성공한 결과는 먼저 그 에이전트의 개인 기록에 저장되고, **출처 표시를 붙인 복사본**이 `shared`에 올라갑니다. 개인 기록은 "누가
만든 원본인지"를 남기는 역할이고, 다음 작업의 프롬프트를 만들 때 지금의 전략들이 검색하는 것은 **`shared`(공동 기록)**입니다. 개인
기록의 내용도 복사본이 `shared`에 있으므로 그 경로로 활용됩니다.

공동 기록 파일 하나를 열어 봅시다.

In [3]:
shared_file = next(memory_files.glob("*.shared.md"))
print(shared_file.read_text(encoding="utf-8"))


## Entry 2026-10-14T11:57:00.582+00:00 aafd71e9693140aaa335964c82a3d38d

> created: 2026-10-14T11:57:00+00:00
> tags: ["codex", "task-result", "task:d6eca787-4879-444a-a503-22f524838dff", "shared-from=21f80bee-120b-4a36-a5d5-a8cb4d355bcc"]

**Task Result**

**Prompt:** README에 설치 방법을 추가해 줘

**Agent:** codex

**Output:**
설치 방법: pip install textstats 명령으로 설치합니다.

**Files changed:** INSTALL.md

## Entry 2026-10-14T11:57:02.012+00:00 680f76c2a93547948c572a1edda8100f

> created: 2026-10-14T11:57:02+00:00
> tags: ["kimi-code", "task-result", "task:b10c06ed-f5c4-466d-bb68-8af321adf4e2", "shared-from=8297cdaa-436e-405c-838d-bad55ceee820"]

**Task Result**

**Prompt:** README에 사용 예시를 추가해 줘

**Agent:** kimi-code

**Output:**
사용 예시: count_words('a b')는 2를 돌려줍니다.

**Files changed:** USAGE.md



**출력 읽는 법** — 기록 하나는 `## Entry …`로 시작하는 한 덩어리입니다.

| 부분 | 뜻 |
|---|---|
| `## Entry <시각> <id>` | 기록 하나의 머리글. 기록마다 겹치지 않는 이름이 붙습니다 |
| `> created:` | 기록된 시각 |
| `> tags:` | 검색·분류용 꼬리표(아래 표) |
| `**Task Result**` 이하 | 실제 내용: 요청(`Prompt`), 에이전트, 답변(`Output`), 바뀐 파일 |

| 태그 | 뜻 |
|---|---|
| `task-result` | 기록의 **종류**. 최종 결과라는 뜻(4장에서 다른 종류를 봅니다) |
| `codex` / `kimi-code` | 결과를 만든 에이전트 |
| `task:<id>` | 어느 작업(task)의 결과인지 |
| `shared-from=<id>` | 이 공동 기록이 **어느 개인 기록에서 복사됐는지**(출처) |

태그만 뽑아 표로 보면 이렇습니다.

In [4]:
import json
import re

def memory_entries(path):
    text = path.read_text(encoding="utf-8")
    for block in re.split(r"(?m)^## Entry ", text)[1:]:
        tags = json.loads(re.search(r"(?m)^> tags: (\[.*\])$", block).group(1))
        prompt = re.search(r"(?m)^\*\*Prompt:\*\* (.*)$", block)
        yield {"tags": tags, "prompt": prompt.group(1) if prompt else ""}

for entry in memory_entries(shared_file):
    kind = next(t for t in entry["tags"] if t in {"task-result", "pipeline-stage", "council-proposal",
                                                 "council-critique", "decompose-plan", "decompose-subtask"})
    agent = next(t for t in entry["tags"] if t in {"codex", "kimi-code", "claude"})
    has_source = any(t.startswith("shared-from=") for t in entry["tags"])
    print(f"{kind:15} {agent:10} 출처표시={has_source}  {entry['prompt']}")

task-result     codex      출처표시=True  README에 설치 방법을 추가해 줘
task-result     kimi-code  출처표시=True  README에 사용 예시를 추가해 줘


## 3. 다음 작업에는 어떻게 붙나요?

이번에는 **설치 방법과 관련된** 새 작업을 시킵니다. 가짜 에이전트가 기록해 둔 "실제로 받은 프롬프트"를 보면, 앞의 두 기록이
어떤 순서로 붙는지 알 수 있습니다.

In [5]:
box.sm("run", "설치 방법 문서를 다듬어 줘", "--agent", "codex", show=False)
received = box.prompts()[-1].read_text(encoding="utf-8")
print(received)

Context from previous work:
[/private/tmp/syncmill-notebook.j_xe1shh/state/memory/files/2026-] **Task Result**

**Prompt:** README에 설치 방법을 추가해 줘

**Agent:** codex

**Output:**
설치 방법: pip install textstats 명령으로 설치합니다.

**Files changed:** INSTALL.md
---
[/private/tmp/syncmill-notebook.j_xe1shh/state/memory/files/2026-] **Task Result**

**Prompt:** README에 사용 예시를 추가해 줘

**Agent:** kimi-code

**Output:**
사용 예시: count_words('a b')는 2를 돌려줍니다.

**Files changed:** USAGE.md

---

Task: 설치 방법 문서를 다듬어 줘


**출력 읽는 법**

- `Context from previous work:` 아래가 메모리에서 찾아 붙인 기록이고, 기록 사이는 `---`로 나뉩니다.
- 맨 끝의 `Task:` 뒤가 여러분이 이번에 준 요청입니다.
- 이번 요청("설치 방법 …")과 **더 관련 있는 설치 기록이 먼저** 붙었습니다. 기록은 요청과의 관련도 순으로 고릅니다.
- 각 기록 앞의 `[…/files/2026-]`는 기록이 들어 있는 파일 위치를 줄여서 보여 준 것입니다.

In [6]:
first_pos = received.index("설치 방법: pip install")
second_pos = received.index("사용 예시: count_words")
print("설치 기록이 먼저 붙었나요?", first_pos < second_pos)
assert first_pos < second_pos

설치 기록이 먼저 붙었나요? True


**한계도 알아 두세요**

- 붙는 기록의 **개수와 분량에는 상한**이 있습니다(관련도 상위 몇 개, 약 3,800바이트 안에서 잘라 붙임). 기록이 많을수록
  각 기록은 짧게 잘립니다.
- 관련도는 단어가 얼마나 겹치는지로 판단합니다(이 샌드박스는 키워드 검색). 요청을 구체적으로 쓸수록 맞는 기록이 붙습니다.
  임베딩 모델을 설정하면 뜻이 비슷한 기록도 찾지만, 모델을 내려받아야 합니다.
- 오래되거나 틀린 기록도 관련도가 높으면 붙을 수 있습니다. 에이전트는 이 맥락을 참고할 뿐이므로, **결과는 언제나 `diff`로
  검토**하세요.

## 4. 전략의 중간 산출물: 종류 태그

02에서 본 `pipeline`·`council`·`decompose`는 최종 결과 말고도 **중간 산출물**을 메모리에 올려서 다음 단계에 넘깁니다. 이
중간 산출물에는 종류 태그가 따로 붙습니다.

| 종류 태그 | 누가 올리나 |
|---|---|
| `task-result` | 모든 전략의 최종 결과 |
| `pipeline-stage` | `pipeline`의 각 단계 결과 |
| `council-proposal` / `council-critique` | `council`의 제안 / 비평 |
| `decompose-plan` / `decompose-subtask` | `decompose`의 실제 실행된 계획 / 하위 작업 결과 |

`pipeline`을 한 번 실행하고 공동 기록의 종류를 다시 봅시다.

In [7]:
box.sm("run", "릴리스 노트를 작성해 줘", "-s", "pipeline", show=False)
kinds = []
for entry in memory_entries(shared_file):
    kind = next(t for t in entry["tags"] if t in {"task-result", "pipeline-stage"})
    agent = next(t for t in entry["tags"] if t in {"codex", "kimi-code"})
    kinds.append(kind)
    print(f"{kind:15} {agent:10} {entry['prompt']}")
assert "pipeline-stage" in kinds

task-result     codex      README에 설치 방법을 추가해 줘
task-result     kimi-code  README에 사용 예시를 추가해 줘
task-result     codex      설치 방법 문서를 다듬어 줘
pipeline-stage  codex      릴리스 노트를 작성해 줘
task-result     kimi-code  릴리스 노트를 작성해 줘


**출력 읽는 법** — 마지막 두 줄이 방금 실행입니다. 1단계(codex)의 결과는 `pipeline-stage`로, 파이프라인 전체의 최종 결과
(마지막 단계 kimi-code)는 `task-result`로 올라갔습니다.

**왜 종류를 나누나요?** 단계마다 **필요한 종류의 기록만** 읽게 하려고입니다. 예를 들어 파이프라인 2단계에 필요한 것은 앞 단계의
결과이지, 관련 있어 보이는 일반 작업 결과가 아닙니다. 그래서 각 전략의 **뒤쪽 단계**는 특정 종류의 기록만 검색합니다.

단, 종류 필터는 "**이번 실행**의 기록만" 고르는 기능은 아닙니다. 지난 실행에서 남은 같은 종류의 기록(예: 예전 파이프라인의
`pipeline-stage`)도 관련도 순위에서 함께 경쟁합니다.

| 단계 | 읽는 종류 |
|---|---|
| `pipeline` 2단계부터 | `pipeline-stage` |
| `decompose` 하위 작업 실행 | `decompose-plan`, `decompose-subtask` |
| `decompose` 종합 | `decompose-subtask` |
| `council` 첫 비평 | `council-proposal` |
| `council` 이후 비평과 종합 | `council-proposal`, `council-critique` |

`route`·`compete`, 그리고 각 전략의 **첫 단계**는 종류를 가리지 않고 관련 기록을 읽습니다.

> **실무에서 의미하는 것**: 여러분이 대화형 세션에서 `shared`에 남긴 메모(7장)는 첫 단계에서는 읽히지만, 위 뒤쪽 단계에서는
> 해당 종류 태그가 붙어 있어야 읽힙니다. 태그가 맞아도 관련도 순위와 분량 상한 때문에 포함이 보장되지는 않습니다.

## 5. 상태 확인과 고장 났을 때

`syncmill status`는 지금 설정된 메모리 백엔드에 연결되는지 보여 줍니다.

In [8]:
box.sm("status")

memtomem: connected
  db_path: /tmp/syncmill-notebook.j_xe1shh/state/memory/memtomem.db


<syncmill status: exit 0>

**출력 읽는 법**: `memtomem: connected`는 기본 백엔드에 연결됐다는 뜻이고, `db_path`는 검색 색인 DB의 위치입니다.
연결에 실패하면 이유가 함께 나옵니다.

**메모리가 고장 나면 작업이 멈출까요?** 메모리 경로를 일부러 쓸 수 없는 곳(`/dev/null/...`)으로 바꿔서 실행해 봅시다.

In [9]:
broken = box.sm(
    "run", "README를 다듬어 줘", "--agent", "codex", show=False,
    extra_env={"SYNCMILL_MEMORY__DB_PATH": "/dev/null/nope.db",
               "SYNCMILL_MEMORY__MEMORY_DIR": "/dev/null/nope"},
)
notice = next(line for line in broken.stdout.splitlines() if "memory unavailable" in line)
print("종료 코드:", broken.returncode)
print("안내:", notice[:110], "…")
print("run id:", box.run_id(broken))
assert broken.returncode == 0

종료 코드: 0
안내: [syncmill] memory unavailable — continuing without shared context (StorageStartupError: Storage startup failed …
run id: 18b39eba-802d-4c55-9261-ae1cb099030d


- 종료 코드 0, run id 발급 — **작업은 정상적으로 끝났습니다.**
- `memory unavailable — continuing without shared context (...)` — 메모리를 쓸 수 없어서 **맥락 없이** 진행했다는 안내입니다.
  터미널에서는 노란색으로 보입니다. 괄호 안이 원인입니다.
- 이 경우 이번 결과도 메모리에 기록되지 않으므로, 다음 작업에 이어지지 않습니다.

이것이 "메모리는 있으면 좋은 부가 기능"이라는 말의 뜻입니다. 노란 안내를 봤다면 `syncmill status`와 `doctor`로 원인을 확인하세요.

## 6. 다른 도구에서 읽기: `syncmill serve`

syncmill이 쌓은 공동 기록을 **Claude Code, Cursor 같은 다른 MCP 클라이언트에서 읽게** 하고 싶다면 `syncmill serve`를 씁니다.
읽기 전용 MCP 서버를 띄워 주는 명령으로, 실제로는 이렇게 등록합니다.

```bash
claude mcp add syncmill-shared -- syncmill serve
```

여기서는 노트북 안에서 MCP 클라이언트를 직접 만들어, Claude Code가 하는 것과 같은 방식으로 `serve`에 붙어 봅니다(MCP
SDK는 syncmill이 이미 쓰고 있어서 추가 설치가 필요 없습니다).

In [10]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(
    command=str(syncmill_executable()), args=["serve"], env=box.env(), cwd=str(box.repo)
)
serve_log = box.state / "serve.log"  # the server's stderr (stdout is the MCP channel)
with serve_log.open("w", encoding="utf-8") as errlog:
    async with stdio_client(server, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = [tool.name for tool in (await session.list_tools()).tools]
            found = await session.call_tool("shared_search", {"query": "설치 방법", "top_k": 2})
            stats = await session.call_tool("shared_stats", {})

print("serve가 stderr에 남긴 안내:", serve_log.read_text(encoding="utf-8").strip())
print("도구:", tools)
print()
print(found.structured_content["result"])
print(stats.structured_content["result"])
assert "설치 방법: pip install" in found.structured_content["result"]

serve가 stderr에 남긴 안내: [syncmill] serving shared memory over stdio (read-only)
도구: ['shared_search', 'shared_read', 'shared_stats', 'board_list', 'board_get', 'board_stats']

1. [score 0.02] /private/tmp/syncmill-notebook.j_xe1shh/state/memory/files/2026-10-14.shared.md (id: b84eef29-d053-4723-8ac3-2659d7c28cf6)
   **Task Result**

**Prompt:** README에 설치 방법을 추가해 줘

**Agent:** codex

**Output:**
설치 방법: pip install textstats 명령으로 설치합니다.

**Files changed:** INSTALL.md
2. [score 0.02] /private/tmp/syncmill-notebook.j_xe1shh/state/memory/files/2026-10-14.shared.md (id: ac364cc0-03aa-4eac-9dde-d79999c65fcb)
   **Task Result**

**Prompt:** 설치 방법 문서를 다듬어 줘

**Agent:** codex

**Output:**
codex (fake) appended one line to NOTES-codex.md.

**Files changed:** NOTES-codex.md
shared: 5 chunks


**출력 읽는 법**

- `serving shared memory over stdio (read-only)` — `serve`가 시작하며 남기는 안내입니다. stdio 방식에서는 표준 출력(stdout)이
  MCP 통신 채널이라, 사람이 읽을 안내는 모두 표준 오류(stderr)로 나옵니다. 여기서는 그 stderr를 파일로 받아 보여 줬습니다.
- 도구는 여섯 개입니다. `shared_*` 세 개는 공동 기록용, `board_*` 세 개는 워크보드용(04에서 봅니다)입니다. **쓰기 도구는
  없습니다.** 다른 도구가 syncmill의 기록을 읽을 수는 있어도 고칠 수는 없습니다.
- `shared_search` 결과의 `[score …]`는 관련도 점수, `(id: …)`는 기록의 id입니다. `shared_read`에 이 id를 넘기면 기록
  전체를 읽습니다.
- `shared_stats`의 `shared: N chunks`는 공동 기록의 개수입니다.
- `serve`는 **`shared`만** 보여 줍니다. 각 에이전트의 개인 기록(`agent-runtime:*`)은 노출하지 않습니다.

> `serve --http`로 HTTP로 열 수도 있지만 **인증이 없습니다.** 내 컴퓨터(기본값)에서만 쓰거나 믿을 수 있는 네트워크 뒤에서만 쓰세요.

## 7. 대화형 세션과 협업하기, 그리고 실무 주의점

`serve`는 **읽기만** 합니다. 여러분이 대화형으로 쓰는 Claude Code가 syncmill에 메모를 **남기고** 싶다면, 그 세션에서
memtomem MCP 도구로 `shared` namespace에 직접 씁니다.

- 쓰기: `mem_add(content=..., namespace="shared")`
- 읽기: `mem_agent_search(query=..., include_shared=True)`

이때 꼭 맞춰야 할 것이 하나 있습니다. **두 쪽이 같은 메모리 저장소를 가리켜야 합니다.** `init`한 프로젝트의 syncmill은
기본적으로 저장소 안 `.syncmill/memory`를 쓰고, 따로 설정하지 않은 memtomem MCP는 `~/.memtomem`을 씁니다. 저장소가 다르면
서로의 기록이 보이지 않습니다. 자세한 협업 방법은 [사용 안내서](../user-guide.ko.md)의 대화형 협업 부분을 보세요.

**실무 주의점**

| 주의할 점 | 이유와 대처 |
|---|---|
| **비밀 정보를 요청에 넣지 마세요** | 요청과 에이전트의 답변이 그대로 Markdown 파일에 남고, 다음 작업 프롬프트에 붙을 수 있습니다 |
| 메모리 폴더를 커밋하지 마세요 | `init`이 만든 `.syncmill/.gitignore`가 무시 처리합니다. 지우지 마세요 |
| 오래된·틀린 기록 | 관련도가 높으면 계속 붙습니다. 기록은 사람이 읽을 수 있는 파일이므로 확인할 수 있습니다 |
| 다른 백엔드 | `mem0`, `langgraph`(로컬 SQLite)도 고를 수 있고, `syncmill memory migration`으로 옮길 수 있습니다 |

## 8. 정리하기

In [11]:
root = box.root
box.cleanup()
print("삭제됨:", not root.exists())

삭제됨: True


## 정리

| 알게 된 것 | 요점 |
|---|---|
| 저장 위치 | `init`한 프로젝트는 `.syncmill/memory/`. 날짜·namespace별 Markdown 파일 |
| 두 namespace | `agent-runtime:<에이전트>`(개인 원본) + `shared`(출처 표시가 붙은 공동 복사본) |
| 태그 | 종류(`task-result`, `pipeline-stage` …), 에이전트, `task:<id>`, `shared-from=<id>` |
| 붙는 방식 | 요청과의 관련도 순, 개수·분량 상한 있음. 뒤쪽 단계는 종류 태그로 거름 |
| 고장 나면 | 노란 안내와 함께 맥락 없이 계속 진행 |
| 다른 도구에서 | `syncmill serve` — `shared`만 읽기 전용으로 공개 |

| 명령 | 뜻 |
|---|---|
| `syncmill status` | 메모리 백엔드 연결 상태 |
| `syncmill serve` | 공동 기록을 읽기 전용 MCP 서버로 열기 |
| `syncmill memory migration plan --from memtomem --to mem0` | 다른 백엔드로 옮길 계획 만들기 |

다음 노트북: **04. 워크보드와 TUI** — 여러 작업을 보드에 쌓아 두고 하나씩 처리하며, 검증된 완료만 `done`으로 기록합니다.